<a href="https://colab.research.google.com/github/Kamindumenula/SE4050-Household-Waste-Classification-System/blob/main/efficientnetB0/efficientnetB0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


#DOWNLOAD & EXTRACT DATASET

if not os.path.exists("SE4050-Household-Waste-Classification-System"):
    !git clone https://github.com/Kamindumenula/SE4050-Household-Waste-Classification-System.git

DATASET_DIR = "/content/dataset-resized"
if not os.path.exists(DATASET_DIR):
    !unzip -q -o SE4050-Household-Waste-Classification-System/dataset/archive.zip -d /content/


# LOAD & SPLIT (80% TRAIN, 10% VAL, 10% TEST)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# Load 80% for training
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Load 20% held-out
val_test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Split the 20% into 10% val and 10% test
val_size = int(len(val_test_ds) * 0.5)
val_ds = val_test_ds.take(val_size)
test_ds = val_test_ds.skip(val_size)

# APPLY COMMON DATA AUGMENTATION (ONLY TO TRAIN DATASET!)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),  # Flips left/right
    layers.RandomRotation(0.15),      # Rotates ±15%
    layers.RandomZoom(0.15),          # Zooms in/out ±15%
    layers.RandomContrast(0.1),       # Slight lighting change
], name="common_data_augmentation")

# We apply augmentation directly to train_ds:
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)


# SPEED OPTIMIZATION (CACHE & PREFETCH)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

classes = sorted(os.listdir(DATASET_DIR))
print(f"Setup Complete!")
print(f"Classes ({len(classes)}): {classes}")
print(f"Batches -> Train: {len(train_ds)} (Augmented) | Val: {len(val_ds)} | Test: {len(test_ds)}")

Cloning into 'SE4050-Household-Waste-Classification-System'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 39 (delta 10), reused 22 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 40.60 MiB | 30.24 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Found 2527 files belonging to 6 classes.
Using 2022 files for training.
Found 2527 files belonging to 6 classes.
Using 505 files for validation.
Setup Complete!
Classes (6): ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Batches -> Train: 64 (Augmented) | Val: 8 | Test: 8
